In [29]:
import dotenv
from web3 import Web3
import json
import os

dotenv.load_dotenv()

True

In [30]:
# Load config
with open("config.json") as f:
    config = json.load(f)

w3 = Web3(Web3.HTTPProvider(config["blockchain"]["rpc_url"]))
print("Connected:", w3.is_connected())

stablecoin_address = config["blockchain"]["stablecoin_address"]
p2p_market_address = config["blockchain"]["p2p_market_address"]

# Minimal ERC-20 ABI — only what we need for setting and checking allowances
erc20_abi = [
    {
        "constant": True,
        "inputs": [
            {"name": "owner", "type": "address"},
            {"name": "spender", "type": "address"}
        ],
        "name": "allowance",
        "outputs": [{"name": "", "type": "uint256"}],
        "type": "function"
    },
    {
        "constant": True,
        "inputs": [{"name": "account", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "", "type": "uint256"}],
        "type": "function"
    },
    {
        "constant": False,
        "inputs": [
            {"name": "spender", "type": "address"},
            {"name": "amount", "type": "uint256"}
        ],
        "name": "approve",
        "outputs": [{"name": "", "type": "bool"}],
        "type": "function"
    }
]

# Adds the stablecoind contract
stablecoin = w3.eth.contract(address=Web3.to_checksum_address(stablecoin_address), abi=erc20_abi)

Connected: True


In [31]:
# Verbindet Wallet für Deployer
account_Deploy = w3.eth.account.from_key(os.environ["DEPLOYER_PRIVATE_KEY"])
print("Connected, chain id:", w3.eth.chain_id)
print("Account:", account_Deploy.address)
print("Balance (ETH):", w3.from_wei(w3.eth.get_balance(account_Deploy.address), "ether"))

Connected, chain id: 11155111
Account: 0x2C4b47689a05f0653637a97a8Db762b764f1653c
Balance (ETH): 0.283680705071633366


In [32]:
# Verbindet Wallet für Trigger
account_trigger = w3.eth.account.from_key(os.environ["TRIGGER_PRIVATE_KEY"])
print("Connected, chain id:", w3.eth.chain_id)
print("Account:", account_trigger.address)
print("Balance (ETH):", w3.from_wei(w3.eth.get_balance(account_trigger.address), "ether"))

Connected, chain id: 11155111
Account: 0x4022d2250AB1E76d3fcbCcf18d39656f4559b83c
Balance (ETH): 0.152825514633798659


In [33]:
# Lädt alle Smart Contracts, um in den Folgenden Abschnitten verwendet zu werden
def load_contract(abi_path: str, address: str):
    with open(abi_path) as f:
        artifact = json.load(f)
    abi = artifact["abi"] if "abi" in artifact else artifact
    return w3.eth.contract(address=Web3.to_checksum_address(address), abi=abi)

bc = config["blockchain"]

oracle_storage = load_contract("abi/OracleStorage.json", bc["oracle_storage_address"])
p2p_market = load_contract("abi/P2PEnergyMarket.json", bc["p2p_market_address"])

oracle_storage.address, p2p_market.address

('0x61769c1C299495194D7Df49030019e613d13a88D',
 '0xB5D7DE4841985feA6a234256B5Adfa4AF1725bF7')

In [34]:
# Authorisiert das Oracle Wallet im OracleStorage Smart Contract
oracle_storage.functions.authorizeOracle(os.getenv("ORACLE_PRIVATE_KEY"))

<Function authorizeOracle(address) bound to ('bb0a0d5667ab20e5f60c80e65afef6188f90a92cf72e689aca335b2bf9dc57b7',)>

In [35]:
# Registiert alle Haushalte im P2P-Markt Smart Contract
for household in config["households"]:
    p2p_market.functions.registerHousehold(household["address"])

In [36]:
# Registriert alle Haushalte im OracleStorage Smart Contract
for household in config["households"]:
    oracle_storage.functions.registerHousehold(household["address"])

In [ ]:
# Gibt die Allowance und den Kontostand jedes Haushalts aus
for household in config["households"]:
    addr = Web3.to_checksum_address(household["address"])
    allowance = stablecoin.functions.allowance(addr, p2p_market_address).call()
    balance = stablecoin.functions.balanceOf(addr).call()
    print(f"{household.get('name', addr)}: allowance={allowance}, balance={balance}")

0x2C4b47689a05f0653637a97a8Db762b764f1653c: allowance=115792089237316195423570985008687907853269984665640564039457584007913129639935, balance=65556100
0xF49153d700AD86CA224f7B2064F541278FE1c320: allowance=115792089237316195423570985008687907853269984665640564039457584007913129639935, balance=5176000
0xC416DDdD40c28eAfE48275f74B4E4Bdee4E3e75b: allowance=115792089237316195423570985008687907853269984665640564039457584007913129639935, balance=29222000


In [ ]:
# Sendet eine Transaktion an die Blockchain und wartet auf die Bestätigung
def send_tx(contract_function, signer=None, gas: int = 200_000):
    signer = signer   # defaults to the notebook's main account
    nonce = w3.eth.get_transaction_count(signer.address, "pending")
    tx = contract_function.build_transaction({
        "from": signer.address,
        "nonce": nonce,
        "gas": gas,
        "chainId": w3.eth.chain_id,
    })
    signed = signer.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    status = "OK" if receipt.status == 1 else "REVERTED"
    print(f"{status} | tx {tx_hash.hex()} | gasUsed {receipt.gasUsed}")
    return receipt

In [ ]:
MAX_UINT256 = 2**256 - 1
ALREADY_APPROVED_THRESHOLD = 2**200  # treat anything this large as "already unlimited"

# Setzt die Allowance auf den maximalen Wert für alle Haushalte im P2P-Markt Smart Contract, falls sie noch nicht ausreichend genehmigt ist
for i, household in enumerate(config["households"], start=1):
    label = household.get("name", household["id"])
    addr = Web3.to_checksum_address(household["address"])
    env_var = f"HOUSEHOLD{i}_PRIVATE_KEY"

    private_key = os.getenv(env_var)
    if not private_key:
        print(f"✗ {label}: {env_var} not set in .env, skipping")
        continue

    hh_account = w3.eth.account.from_key(private_key)
    if hh_account.address.lower() != addr.lower():
        print(f"✗ {label}: {env_var} resolves to {hh_account.address}, "
              f"but config.json has {addr} — skipping")
        continue

    current_allowance = stablecoin.functions.allowance(addr, p2p_market_address).call()
    if current_allowance >= ALREADY_APPROVED_THRESHOLD:
        print(f"✓ {label}: already approved ({current_allowance})")
        continue

    print(f"→ {label}: approving (current allowance {current_allowance})")
    send_tx(
        stablecoin.functions.approve(p2p_market_address, MAX_UINT256),
        signer=hh_account,
        gas=100_000,
    )

✓ house_01: already approved (115792089237316195423570985008687907853269984665640564039457584007913129639935)
✓ house_02: already approved (115792089237316195423570985008687907853269984665640564039457584007913129639935)
✓ house_03: already approved (115792089237316195423570985008687907853269984665640564039457584007913129639935)
